# Grid search

O `GridSearchCV` do Scikit-Learn é uma ferramenta utilizada para otimizar hiperparâmetros de modelos de machine learning. Hiperparâmetros são parâmetros do modelo que não são aprendidos diretamente pelos algoritmos de aprendizado, mas que precisam ser definidos antes do processo de treino. Exemplos comuns incluem a profundidade máxima de uma árvore de decisão ou a taxa de aprendizado de um modelo de gradient boosting. Encontrar os valores corretos desses hiperparâmetros pode impactar significativamente a performance do modelo.

O `GridSearchCV` automatiza esse processo, testando exaustivamente todas as combinações possíveis de valores fornecidos para os hiperparâmetros, criando uma grade (ou "grid") de opções. Ele faz isso utilizando validação cruzada (cross-validation), o que significa que o conjunto de dados é dividido em múltiplas partes (folds) e o desempenho do modelo é avaliado em diferentes subdivisões. Dessa forma, o processo minimiza o risco de overfitting e garante que o modelo generalize melhor para dados não vistos.

Após testar todas as combinações, o `GridSearchCV` retorna o conjunto de hiperparâmetros que apresentou o melhor desempenho com base na métrica de avaliação definida, como acurácia, F1-score, entre outros.

Em resumo, o `GridSearchCV` é uma abordagem sistemática e eficiente para encontrar os melhores hiperparâmetros de um modelo, evitando o trabalho manual de testar diferentes combinações e proporcionando resultados mais robustos.

https://scikit-learn.org/stable/modules/grid_search.html#grid-search

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    PowerTransformer,
    StandardScaler,
    QuantileTransformer,
)

PASTA_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PASTA_PROJETO) not in sys.path:
    sys.path.append(str(PASTA_PROJETO))

from src.config import DADOS_DIABETES_CATEGORIZADOS
from src.modelos import (
    RANDOM_STATE,
    grid_search_cv_regressor,
    organiza_resultados,
    treinar_e_validar_modelo_regressao,
)

sns.set_theme(palette="bright")

In [2]:
df = pd.read_parquet(DADOS_DIABETES_CATEGORIZADOS)

df.head()

,idade,sexo,imc,pressao_media,colesterol_total,ldl,hdl,triglicerides,glicose,target,colesterol_hdl_cat
0,59,2,32.099998,101.0,157,93.199997,38.0,4.8598,87,151,4-5
1,48,1,21.600000,87.0,183,103.199997,70.0,3.8918,69,75,2-3
2,72,2,30.500000,93.0,156,93.599998,41.0,4.6728,85,141,4-5
3,24,1,25.299999,84.0,198,131.399994,40.0,4.8903,89,206,4-5
4,50,1,23.000000,101.0,192,125.400002,52.0,4.2905,80,135,4-5


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   idade               442 non-null    int8    
 1   sexo                442 non-null    int8    
 2   imc                 442 non-null    float32 
 3   pressao_media       442 non-null    float32 
 4   colesterol_total    442 non-null    int16   
 5   ldl                 442 non-null    float32 
 6   hdl                 442 non-null    float32 
 7   triglicerides       442 non-null    float32 
 8   glicose             442 non-null    int8    
 9   target              442 non-null    int16   
 10  colesterol_hdl_cat  442 non-null    category
dtypes: category(1), float32(5), int16(2), int8(3)
memory usage: 12.3 KB


In [4]:
X = df.drop(columns="target")
y = df["target"]

In [5]:
colunas_power_transform = ["imc", "ldl", "hdl", "colesterol_total"]

coluna_target = ["target"]

coluna_ordinal_encoder = ["colesterol_hdl_cat"]

coluna_one_hot_encoder = ["sexo"]

colunas_standard_scaler = [
    coluna
    for coluna in df.columns
    if coluna not in colunas_power_transform + coluna_target + coluna_ordinal_encoder + coluna_one_hot_encoder
]

colunas_standard_scaler

['idade', 'pressao_media', 'triglicerides', 'glicose']

In [6]:
categorias_ordinal_encoder = [
    ["2-3", "4-5", "6+"],
]

In [7]:
preprocessamento_categoricas = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ],
    remainder="passthrough",
)

preprocessamento_simples = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
        ("standard_scaler", StandardScaler(), X.columns.difference(coluna_ordinal_encoder + coluna_one_hot_encoder)),
    ],
    remainder="passthrough",
)

preprocessamento_completo = ColumnTransformer(
    [
        ("power_transform", PowerTransformer(method="box-cox"), colunas_power_transform),
        ("standard_scaler", StandardScaler(), colunas_standard_scaler),
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ]
)

target_transformer = QuantileTransformer(n_quantiles=20, output_distribution="normal")